# Module 1.1: Vectorized NumPy on 4-20mA Industrial Telemetry
### 🇵🇱 Wektoryzacja NumPy w Przetwarzaniu Pętli Prądowych 4-20mA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m01_l01_numpy_4_20ma_telemetry.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Convert raw 16-bit ADC integer counts into calibrated 4-20mA current loop telemetry with vectorized NumPy, eliminating Python loop bottlenecks.  
> 🇵🇱 **Streszczenie:** Przeliczaj surowe 16-bitowe liczby z przetwornika ADC na pętlę prądową 4-20mA za pomocą wektoryzacji NumPy, eliminując powolne pętle w Pythonie.

- **Estimated Time:** 20 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **4-20mA Current Loops**: The global gold standard for analog industrial instrumentation. Current is immune to lead resistance and voltage drops over hundreds of meters.
2. **ADC Conversion Equation**: A 16-bit ADC maps $0-65535$ raw integer ticks to $4.0 - 20.0\text{ mA}$:
   $$I = 4.0 + 16.0 \cdot \frac{\text{ADC} - \text{ADC}_{min}}{\text{ADC}_{max} - \text{ADC}_{min}}$$
3. **The 'Live Zero' Principle**: $4.0\text{ mA}$ represents $0\%$ physical process value. If a wire snaps, current drops to $0.0\text{ mA}$. Any reading below $3.6\text{ mA}$ is an immediate broken-wire alarm.
4. **Vectorized NumPy Operations**: Processing millions of PLC ticks via vectorized boolean masks executes in C-speed SIMD registers, achieving 50x-100x speedup over standard Python `for` loops.


In [ ]:
# 1. Simulate 1,000,000 16-Bit ADC Telemetry Samples with Live Zero Faults
np.random.seed(42)
n_samples = 1_000_000

# 16-bit unsigned ADC integer range
ADC_MIN = 0
ADC_MAX = 65535

# True process pressure in a chemical reactor: 0 to 100 bar
# Normal operational baseline: 45.0 bar +/- 6.0 bar
true_pressure = np.random.normal(loc=45.0, scale=6.0, size=n_samples)
true_pressure = np.clip(true_pressure, 0.0, 100.0)

# Ideal current: 4 mA = 0 bar, 20 mA = 100 bar
ideal_current = 4.0 + (true_pressure / 100.0) * 16.0

# Add electromagnetic sensor noise
noisy_current = ideal_current + np.random.normal(0, 0.12, size=n_samples)

# Quantize to 16-bit integer ADC counts
raw_adc = np.round(((noisy_current - 4.0) / 16.0) * (ADC_MAX - ADC_MIN) + ADC_MIN).astype(np.int32)
raw_adc = np.clip(raw_adc, ADC_MIN, ADC_MAX)

# Inject physical faults:
# - Severed cable / broken wire: current drops to 0.0 mA (ADC ~ 0)
# - Sensor short circuit / over-range: current spikes to 22.5 mA (ADC pegged at 65535)
fault_broken_wire = np.random.choice(n_samples, size=500, replace=False)
raw_adc[fault_broken_wire] = 120  # ~ 0.03 mA

fault_saturation = np.random.choice(n_samples, size=300, replace=False)
raw_adc[fault_saturation] = 65535  # Pegged max

print(f"✓ Generated {n_samples:,} 16-bit ADC samples with injected electrical faults.")


In [ ]:
# 2. Vectorized Engineering Conversion & Live Zero Masking
import time

# Vectorized conversion to mA and Physical Bar
t0 = time.perf_counter()
current_ma = 4.0 + 16.0 * (raw_adc - ADC_MIN) / (ADC_MAX - ADC_MIN)
calc_pressure = (current_ma - 4.0) / 16.0 * 100.0

# Vectorized Live Zero Fault Classification
mask_broken_wire = current_ma < 3.6    # Broken wire / disconnected loop
mask_saturation  = current_ma > 20.8   # Over-range / short circuit
mask_normal      = (~mask_broken_wire) & (~mask_saturation)
t_vectorized = time.perf_counter() - t0

print(f"Vectorized NumPy Pipeline: {t_vectorized*1000:.2f} ms for {n_samples:,} samples")
print(f" - Normal operational samples : {mask_normal.sum():,}")
print(f" - Broken wire alarms (<3.6mA) : {mask_broken_wire.sum():,}")
print(f" - Saturation alarms (>20.8mA) : {mask_saturation.sum():,}")

# Visual Inspection of Telemetry Window
sample_window = slice(1000, 1500)
t_axis = np.arange(500)

plt.figure(figsize=(14, 5))
plt.plot(t_axis, calc_pressure[sample_window], color='#0284c7', linewidth=1.5, label='Calculated Pressure [bar]')
plt.axhline(0.0, color='#64748b', linestyle='--', alpha=0.6, label='0 bar Process Zero (4.0 mA)')
plt.axhline(100.0, color='#ef4444', linestyle='--', alpha=0.6, label='100 bar Max Scale (20.0 mA)')

# Highlight detected wire break anomalies
wire_breaks_in_window = np.where(mask_broken_wire[sample_window])[0]
if len(wire_breaks_in_window) > 0:
    plt.scatter(wire_breaks_in_window, calc_pressure[sample_window][wire_breaks_in_window],
                color='#ef4444', s=80, marker='x', linewidth=2, label='ALARM: Broken Wire (<3.6 mA)')

plt.title("Industrial 4-20mA Telemetry Pipeline with Live Zero Anomaly Detection", fontsize=13, fontweight='bold')
plt.xlabel("Telemetry Sample Index")
plt.ylabel("Pressure [bar]")
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Map 16-bit unsigned ADC counts [0, 65535] linearly to [4.0, 20.0] mA range *(🇵🇱 Przeskaluj 16-bitowe rejestry ADC [0, 65535] do zakresu pętli prądowej [4.0, 20.0] mA)*
- [ ] **Task 2:** Benchmark vectorized NumPy operations against native Python loops on 1M samples *(🇵🇱 Zmierz czas wektoryzacji NumPy w porównaniu do pętli for na 1 milionie próbek)*
- [ ] **Task 3:** Detect line breaks (< 3.8mA) and short circuits (> 20.5mA) using boolean bitmasks *(🇵🇱 Zidentyfikuj przerwanie przewodu (< 3.8mA) i zwarcie (> 20.5mA) za pomocą masek binarnych)*


In [ ]:
# Exercise Challenge:
# 1. Assert that exactly 500 broken-wire faults were identified by the boolean mask.
# 2. Verify that vectorized compute time is under 50 milliseconds.

assert mask_broken_wire.sum() == 500, f"Expected 500 wire breaks, got {mask_broken_wire.sum()}"
assert t_vectorized < 0.05, f"Vectorized latency too high: {t_vectorized*1000:.1f}ms"
print("✓ Vectorized telemetry validation test passed with zero errors!")
